<a href="https://colab.research.google.com/github/maick-code/airf-multilingual-tokenizer-challenge/blob/arena/01a09f96-airf-multilingual-tokenizer-ch/submissions/maick-dane-nkou/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Research Foundations Multilingual Tokenization Challenge

## Maick Dane Nkou — Final Submission

Six languages, one vocabulary, a maximum of **10,000 tokens**.
This notebook follows the official starter's flow: setup, data inspection,
training, scoring, comparison, export, and final checklist.

### What This Notebook Provides

- Loads **only** the official public training and validation splits.
- Inspects the six-language training distribution.
- Trains the selected **`weights-yo-am4`** recipe from scratch in a single run.
- Uses the official checker, including **both** reconstruction and guardrail penalties.
- Verifies exact reconstruction, including spaces, case and diacritics.
- Exports a verified `tokenizer.json` and its matching metadata and README.

**Current artifact:** participant-reported validation score **1.963189**, 100%
reconstruction, zero UNK and guardrail penalty. This is a recorded result, not a
claim that the notebook has just been rerun or that hidden-test scores match.

The original optimization notebook is preserved at commit
[`17d3495`](https://github.com/maick-code/airf-multilingual-tokenizer-challenge/blob/17d34954a86e9baabb46658478ac7a0160e3a04d/submissions/maick-dane-nkou/notebook.ipynb).
It selected this recipe using validation only. This final notebook does **not**
repeat that search or download any trained tokenizer, vocabulary or merge table.

**Run all in Colab on CPU.** Retraining may resolve BPE merge ties differently;
measure each generated artifact rather than assuming an identical score/hash.

# Getting Started

Install only missing dependencies, like the starter notebook. The runtime must
use `tokenizers==0.22.1`. An existing uv environment is supported without requiring
pip when the dependencies are already installed. Dependencies and public data
require network access; no secrets, private test data or external corpus is used.

In [ ]:
# Runtime setup — ordinary Python, usable in Colab and a local notebook kernel.
import importlib.util
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

TOKENIZERS_VERSION = "0.22.1"


def needs_install(package, *, exact=None, major=None):
    try:
        found = version(package)
    except PackageNotFoundError:
        return True
    if exact is not None:
        return found != exact
    if major is not None:
        return found.split(".", 1)[0] != str(major)
    return False


requirements = [
    ("tokenizers", f"tokenizers=={TOKENIZERS_VERSION}", {"exact": TOKENIZERS_VERSION}),
    ("datasets", "datasets>=4.0,<5", {"major": 4}),
    ("pandas", "pandas", {}),
    ("matplotlib", "matplotlib", {}),
    ("PyYAML", "PyYAML==6.0.2", {"exact": "6.0.2"}),
]
missing = [requirement for name, requirement, constraint in requirements
           if needs_install(name, **constraint)]
if missing:
    if importlib.util.find_spec("pip") is not None:
        command = [sys.executable, "-m", "pip", "install", "-q", *missing]
    elif shutil.which("uv"):
        command = ["uv", "pip", "install", "--python", sys.executable, *missing]
    else:
        raise RuntimeError("Install dependencies with uv sync --dev --group data, then restart the kernel")
    subprocess.run(command, check=True)
else:
    print("Dependencies already available at the required versions.")

import tokenizers
if tokenizers.__version__ != TOKENIZERS_VERSION:
    raise RuntimeError("Restart the runtime after installing tokenizers==0.22.1")
print("tokenizers version:", tokenizers.__version__)

In [ ]:
# Competition configuration and portable output locations.
import hashlib
import importlib.util
import json
import math
import tempfile
import time
import unicodedata
import urllib.request
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from datasets import load_dataset
from tokenizers import Regex, Tokenizer, decoders, models, pre_tokenizers, trainers

GITHUB_REPO = "aims-ai-research-foundations/airf-multilingual-tokenizer-challenge"
HF_DATASET = "Similoluwa/african-multilingual-tokenizer-challenge"
HF_REVISION = "v1.0.0"
LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
LANGUAGE_NAMES = {"en": "English", "fr": "French", "ha": "Hausa",
                  "sw": "Swahili", "yo": "Yoruba", "am": "Amharic"}
SCORED_LANGUAGES = ("ha", "sw", "yo", "am")
MAX_VOCAB_SIZE = 10_000
MAX_FILE_BYTES = 20 * 1024 * 1024
TEAM_SLUG = "maick-dane-nkou"
RECIPE = "weights-yo-am4"
MIN_FREQUENCY = 5
BOOST = {"ha": 2, "sw": 2, "yo": 4, "am": 4}  # English/French remain x1
MIN_GUARDRAIL_HEADROOM = 0.05  # our selection policy, NOT an official validity requirement

# Pin the official checker used for the reported result, not a mutable/stale utils.py.
OFFICIAL_COMMIT = "75578f2400c39b1f8e31ce7e7104b37fbc470d11"
OFFICIAL_UTILS_SHA256 = "1727de34136097eb48addabf90501589bdfefa31c20e201bab53c38f2f7c9688"
RAW_URL = f"https://raw.githubusercontent.com/{GITHUB_REPO}/{OFFICIAL_COMMIT}"
SUBMITTED_SHA256 = "1519895eace8680d2752b333f5efd82f80ade21bcd6210704ec17de55dafe035"


def local_file(relative_path):
    """Find a repository file whether the kernel starts at root or in this team folder."""
    for root in (Path.cwd(), *Path.cwd().parents):
        candidate = root / relative_path
        if candidate.is_file():
            return candidate
    return None


project_file = local_file("pyproject.toml")
WORKSPACE = project_file.parent if project_file is not None else Path.cwd()
# Never put reports/caches next to the four submitted files in a repository checkout.
RUN_DIR = WORKSPACE / "artifacts" / "final-weights-yo-am4"
RUN_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load the official profile_submission helper, like the starter, but hash-verified.
def ensure_utils():
    """Reuse only an exact matching helper; otherwise download the pinned official file."""
    destination = RUN_DIR / "official_utils.py"
    candidates = [destination, local_file("starter/utils.py")]
    helper = None
    for path in candidates:
        if path is not None and path.is_file():
            content = path.read_bytes()
            if hashlib.sha256(content).hexdigest() == OFFICIAL_UTILS_SHA256:
                helper = content
                break
    if helper is None:
        with urllib.request.urlopen(f"{RAW_URL}/starter/utils.py", timeout=60) as response:
            helper = response.read()
    if hashlib.sha256(helper).hexdigest() != OFFICIAL_UTILS_SHA256:
        raise RuntimeError("Official checker SHA-256 mismatch; do not continue")
    destination.write_bytes(helper)
    spec = importlib.util.spec_from_file_location("submission_official_utils", destination)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    if module.REQUIRED_TOKENIZERS_VERSION != TOKENIZERS_VERSION or module.RECONSTRUCTION_PENALTY != 3.0:
        raise RuntimeError("Unexpected official checker contract")
    return module


official_utils = ensure_utils()
profile_submission = official_utils.profile_submission
print("Official checker:", OFFICIAL_COMMIT)

## Load the Dataset

Inspect the training distribution before training. There are 40,000 training
and 4,000 validation rows per language. **Original strings are never stripped,
lowercased or normalized.** Only `train` is given to the trainer; validation is
reserved for evaluation. There is no reduced-data mode for final export.

In [ ]:
def load_competition_data(split):
    """Load an official public split as a DataFrame, as in the starter notebook."""
    if split not in {"train", "validation"}:
        raise ValueError("Only train and validation are public")
    dataset = load_dataset(HF_DATASET, split=split, revision=HF_REVISION,
                           cache_dir=str(RUN_DIR / "dataset-cache"))
    frame = dataset.to_pandas()[["language", "text"]].reset_index(drop=True)
    frame.attrs["dataset_fingerprint"] = dataset._fingerprint
    expected_per_language = 40_000 if split == "train" else 4_000
    if Counter(frame.language) != dict.fromkeys(LANGUAGES, expected_per_language):
        raise ValueError(f"Unexpected {split} language counts")
    if not all(isinstance(text, str) and text.split() for text in frame.text):
        raise ValueError(f"Invalid text in {split}")
    return frame


train = load_competition_data("train")
validation = load_competition_data("validation")
print(f"Train rows:      {len(train):,}")
print(f"Validation rows: {len(validation):,}")

In [ ]:
# Equal row counts do not imply equal amounts of text.
summary = train.assign(characters=train.text.str.len()).groupby("language").agg(
    rows=("text", "size"), characters=("characters", "sum"), mean_length=("characters", "mean"))
summary["share_of_characters"] = summary.characters / summary.characters.sum()
summary["training_repeat"] = [BOOST.get(lang, 1) for lang in summary.index]
print(summary.round({"mean_length": 1, "share_of_characters": 3}))

sizes = summary.characters.rename(index=LANGUAGE_NAMES).sort_values() / 1e6
fig, ax = plt.subplots(figsize=(7.2, 3.4))
ax.barh(sizes.index, sizes.values, color="#4878a8")
ax.set_xlabel("Characters in the training split (millions)")
ax.set_title("Training distribution before language weighting")
plt.tight_layout()
plt.show()

for language in LANGUAGES:
    text = train.loc[train.language == language, "text"].sample(1, random_state=41).iloc[0]
    print(f"[{language}] {LANGUAGE_NAMES[language]}: {text[:120]}")

# Train the Final Tokenizer

## Selected Approach: Lossless BPE with Punctuation-Attached Words

We retain the winning configuration, not the starter's deliberately weak
word-level or character-level baselines:

- BPE with **10,000** entries and minimum frequency **5**.
- `Split(Regex(r" ?\S+|\s+"), behavior="isolated")`, followed by
  `ByteLevel(add_prefix_space=False, use_regex=False)`.
- Matching `ByteLevel` decoder and the complete 256-symbol byte alphabet.
- No normalizer, special tokens or post-processor.
- Balanced round-robin training; ha/sw repeated **x2**, yo/am **x4**, en/fr **x1**.

Unlike `WhitespaceSplit`, the isolated split **retains every separator**. The
byte alphabet is an encoding definition, not a pretrained vocabulary. All
vocabulary learning and BPE merges come from this participant's training code
on the official **train** split.

In [ ]:
def new_tokenizer():
    """Build the exact reversible space_word pipeline used by weights-yo-am4."""
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
        pre_tokenizers.Split(Regex(r" ?\S+|\s+"), behavior="isolated"),
        pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False),
    ])
    tokenizer.decoder = decoders.ByteLevel()
    return tokenizer


def corpus_iterator(train_by_lang):
    """Stable language order; use only unmodified training strings."""
    iterators = {lang: iter(train_by_lang[lang]) for lang in LANGUAGES}
    active = list(LANGUAGES)
    while active:
        for lang in active.copy():
            try:
                text = next(iterators[lang])
            except StopIteration:
                active.remove(lang)
                continue
            for _ in range(BOOST.get(lang, 1)):
                yield text


def train_final(texts, *, vocab_size=MAX_VOCAB_SIZE, min_frequency=MIN_FREQUENCY):
    """Train from scratch. Smaller parameters are for disposable regression tests only."""
    tokenizer = new_tokenizer()
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size, min_frequency=min_frequency, special_tokens=[],
        initial_alphabet=sorted(pre_tokenizers.ByteLevel.alphabet()), show_progress=True,
    )
    tokenizer.train_from_iterator(texts, trainer=trainer)
    assert tokenizer.normalizer is None
    assert tokenizer.get_vocab_size(with_added_tokens=True) <= vocab_size
    assert set(pre_tokenizers.ByteLevel.alphabet()) <= set(tokenizer.get_vocab())
    return tokenizer

## Exact Reconstruction and Submission Gates

The official `valid` flag alone does not exclude reconstruction penalties.
Our export checks additionally require zero unknown tokens, exact reconstruction,
zero guardrail penalty and the 5% margin used when choosing this recipe.
The 5% margin is our conservative choice, **not** an official submission rule.
A failed margin check means “review this retrained candidate”, not disqualification.

The small regression model below is discarded. Its synthetic examples and
vocabulary never enter the final training run.

In [ ]:
def assert_exact_roundtrip(tokenizer, texts, *, batch_size=512):
    failures = 0
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        ids = [enc.ids for enc in tokenizer.encode_batch(batch, add_special_tokens=False)]
        restored = tokenizer.decode_batch(ids, skip_special_tokens=False)
        skipped = tokenizer.decode_batch(ids, skip_special_tokens=True)
        failures += sum(original != decoded or original != decoded_skip
                        for original, decoded, decoded_skip in zip(batch, restored, skipped, strict=True))
    if failures:
        raise ValueError(f"Export blocked: {failures}/{len(texts)} rows are not reconstructed exactly")


def assert_submission_ready(report, expected_rows=24_000):
    if not report.get("valid"):
        raise ValueError(f"Official validity check failed: {report.get('errors')}")
    if report.get("rows") != expected_rows:
        raise ValueError("Incomplete validation evaluation")
    if report.get("vocab_size") != MAX_VOCAB_SIZE:
        raise ValueError("Expected a 10,000-entry vocabulary")
    if report.get("reconstruction") != 1.0 or report.get("lossy_rows") != 0:
        raise ValueError("Reconstruction must be 100%")
    if report.get("reconstruction_penalty") != 0.0:
        raise ValueError("Nonzero reconstruction penalty")
    if set(report.get("fertility", {})) != set(LANGUAGES):
        raise ValueError("Missing validation language")
    if set(report.get("unknown_rate", {})) != set(LANGUAGES):
        raise ValueError("Missing unknown-token measurements")
    if any(value != 0.0 for value in report["unknown_rate"].values()):
        raise ValueError("Unknown tokens were emitted")
    for key in ("score", "guardrail_budget", "guardrail_penalty"):
        if not math.isfinite(report[key]) or report[key] < 0:
            raise ValueError(f"Invalid metric: {key}")
    base = sum(report["penalised"][lang] for lang in SCORED_LANGUAGES) / 4
    total = base + report["guardrail_penalty"] + report["reconstruction_penalty"]
    if not math.isclose(report["score"], total, rel_tol=1e-12, abs_tol=1e-12):
        raise ValueError("Full score is missing a penalty")
    budget = report["guardrail_budget"]
    if budget <= 0 or report["guardrail_penalty"] != 0:
        raise ValueError("Review the English/French guardrail before export")
    headroom = min((budget - report["fertility"][lang]) / budget for lang in ("en", "fr"))
    if not math.isfinite(headroom) or headroom < MIN_GUARDRAIL_HEADROOM - 1e-12:
        raise ValueError("Candidate falls below our 5% guardrail margin; review before export")


SMOKE_TEXTS = list(official_utils.SMOKE_TEXTS.values())
ROUNDTRIP_CASES = SMOKE_TEXTS + [
    "", " ", "   ", "\t\n\r\n", "  Hello  WORLD!\tNext\nline.  ",
    "[UNK] [CLS] [SEP] <0xFF>", "é e\u0301 Ì I\u0300", "👩🏿‍💻 🌍 中文 العربية",
    "a\u00a0b\u2003c\u200bd", "\x00\x01\x7f\ufeff\U0010ffff", "don't l’amour — … ።",
] + [unicodedata.normalize("NFD", text) for text in SMOKE_TEXTS]


def run_regression_tests():
    from tokenizers import normalizers
    mini = train_final(SMOKE_TEXTS, vocab_size=512, min_frequency=1)
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "tokenizer.json"
        mini.save(str(path))
        loaded = Tokenizer.from_file(str(path))
        assert_exact_roundtrip(loaded, ROUNDTRIP_CASES)
        for mutation in ("lowercase", "wrong_decoder", "whitespace_split"):
            broken = Tokenizer.from_str(loaded.to_str())
            if mutation == "lowercase":
                broken.normalizer = normalizers.Lowercase()
            elif mutation == "wrong_decoder":
                broken.decoder = decoders.ByteFallback()
            else:
                broken.pre_tokenizer = pre_tokenizers.WhitespaceSplit()
            try:
                assert_exact_roundtrip(broken, ROUNDTRIP_CASES)
            except ValueError:
                pass
            else:
                raise AssertionError(f"Lossy mutation accepted: {mutation}")
    print(f"Regression tests passed ({len(ROUNDTRIP_CASES)} strict round-trip cases).")


run_regression_tests()

In [ ]:
# One final training run; validation and regression examples are never supplied.
train_by_lang = {
    lang: train.loc[train.language == lang, "text"].tolist() for lang in LANGUAGES
}
started = time.perf_counter()
tokenizer = train_final(corpus_iterator(train_by_lang))
training_seconds = time.perf_counter() - started
if tokenizer.get_vocab_size(with_added_tokens=True) != MAX_VOCAB_SIZE:
    raise ValueError("Unexpected vocabulary size after full training")
print(f"Training completed in {training_seconds:.1f} s")

candidate_dir = RUN_DIR / "candidate"
candidate_dir.mkdir(parents=True, exist_ok=True)
candidate_path = candidate_dir / "tokenizer.json"
tokenizer.save(str(candidate_path), pretty=True)
if candidate_path.stat().st_size > MAX_FILE_BYTES:
    raise ValueError("Tokenizer exceeds the 20 MiB limit")
# Always test the file that will actually be submitted, not just an in-memory model.
tokenizer = Tokenizer.from_file(str(candidate_path))
assert_exact_roundtrip(tokenizer, ROUNDTRIP_CASES)
assert_exact_roundtrip(tokenizer, validation.text.tolist())
print("All validation strings reconstructed exactly.")

# Score the Tokenizer

For each language, $S_l = F_l + 100 U_l$, where $F_l$ is tokens per word and
$U_l$ is unknown tokens per word. The **full** score is:

$$\frac{S_{ha}+S_{sw}+S_{yo}+S_{am}}{4}
+ P_{\mathrm{guardrail}} + P_{\mathrm{reconstruction}}.$$

We call `profile_submission` directly, rather than copy the older starter's
base-only scoring function. English/French overages and reconstruction loss
must not be omitted. The checker pin records the scoring version; the organizers
may update it. Hidden-test results can differ from validation.

In [ ]:
# Same official entry point as the starter; includes every score component.
report = profile_submission(candidate_path, data=validation, repeats=3)
assert_submission_ready(report, expected_rows=len(validation))
base_score = sum(report["penalised"][lang] for lang in SCORED_LANGUAGES) / 4
measured_sha256 = hashlib.sha256(candidate_path.read_bytes()).hexdigest()
print(f"Base score:              {base_score:.6f}")
print(f"Guardrail penalty:       {report['guardrail_penalty']:.6f}")
print(f"Reconstruction penalty:  {report['reconstruction_penalty']:.6f}")
print(f"Full validation score:   {report['score']:.6f}")
print("Strict reconstruction: 100%")
print("Candidate SHA-256:", measured_sha256)
print("Same bytes as the previously submitted artifact:", measured_sha256 == SUBMITTED_SHA256)

results = pd.DataFrame([
    {"language": LANGUAGE_NAMES[lang], "tokens/word": report["fertility"][lang],
     "UNK rate": report["unknown_rate"][lang], "language score": report["penalised"][lang]}
    for lang in LANGUAGES
]).set_index("language")
print(results.round(6))
fig, ax = plt.subplots(figsize=(7.2, 3.4))
results["tokens/word"].plot.bar(ax=ax, color="#4878a8")
ax.axhline(report["guardrail_budget"], color="#be6b35", linestyle="--",
           label="English/French guardrail budget")
ax.set_ylabel("Tokens per word (lower is better)")
ax.set_title("Final tokenizer — measured validation fertility")
ax.legend()
plt.tight_layout()
plt.show()

## Compare with Previous Experiments

These are **participant-reported historical validation results**, not new
executions of baselines in this notebook:

| Candidate | Full score ↓ | Reconstruction | EN/FR headroom | Outcome |
| --- | ---: | ---: | ---: | --- |
| Lossless ByteLevel reference | 2.023252 | 100% | 5.2% | Previous model |
| `train-space_word-b3` | 1.948827 | 100% | 4.8% | Below chosen 5% safety margin |
| `weights-b2` | 1.967755 | 100% | 9.5% | Eligible |
| `weights-b4` | 1.939667 | 100% | 1.8% | Below chosen 5% safety margin |
| **`weights-yo-am4`** | **1.963189** | **100%** | **5.3%** | **Selected** |

The 5% margin is a participant policy, not an official validity rule. All rows
above had zero reported guardrail penalty. Our selected model sacrifices some
validation score for more context-language margin. This is not a guarantee on
hidden data. The old lowercase model's 1.7440 was **base-only**; its full score
was 4.7426 because of reconstruction loss. Do not compare it as a full score.

The original seven-run search and the exact submitted artifact remain accessible
in Git at `17d34954a86e9baabb46658478ac7a0160e3a04d`.

# Prepare Your Submission

Like the starter, export a single `tokenizer.json` and check it with
`profile_submission`. We also create matching metadata and a measured README.
The export is kept under ignored `artifacts/`, **not** written over your existing
submission automatically. Review the newly measured score before replacing a
previous artifact. A retraining run is not assumed to reproduce the exact hash.

If you only reformatted this notebook, there is **no need** to replace the already
validated tokenizer. Keep the current model and its measured metadata together.

In [ ]:
def export_submission(path, measured, expected_sha256, validation_texts, output_dir):
    """Export only an unchanged, fully checked file; never silently swap models."""
    path = Path(path)
    output_dir = Path(output_dir)
    assert_submission_ready(measured, expected_rows=24_000)
    if len(validation_texts) != 24_000:
        raise ValueError("All 24,000 validation strings are required for export")
    payload = path.read_bytes()
    if len(payload) > MAX_FILE_BYTES:
        raise ValueError("Tokenizer exceeds the 20 MiB limit")
    if hashlib.sha256(payload).hexdigest() != expected_sha256:
        raise ValueError("Candidate changed since evaluation; evaluate it again")
    assert_exact_roundtrip(Tokenizer.from_str(payload.decode("utf-8")), validation_texts)
    # No output files are changed until the gates above have passed.
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "tokenizer.json").write_bytes(payload)
    metadata = {
        "team": "Maick Dane Nkou", "members": ["Maick Dane Nkou"],
        "affiliation": "AIMS SOUTH AFRICA",
        "approach": (f"Lossless BPE 10000, space_word boundaries, no normalization; "
                     f"official train only, ha/sw x2 and yo/am x4; validation "
                     f"{measured['score']:.6f}, reconstruction 100%."),
    }
    (output_dir / "metadata.yml").write_text(yaml.safe_dump(metadata, sort_keys=False), encoding="utf-8")
    lines = [
        "# Maick Dane Nkou — lossless BPE", "", "## Recipe", "",
        "- weights-yo-am4: space_word boundaries, no normalizer or special tokens",
        "- BPE 10,000; minimum frequency 5; full byte alphabet; ByteLevel decoder",
        "- Official train only; balanced round-robin; ha/sw x2, yo/am x4, en/fr x1",
        f"- Dataset: `{HF_DATASET}` @ `{HF_REVISION}`",
        "- No pretrained tokenizer, published vocabulary/merge table or external corpus",
        "", "## Measured validation results (24,000 rows; not hidden-test scores)", "",
        "| Language | Tokens/word | UNK rate |", "| --- | ---: | ---: |",
    ]
    for lang in LANGUAGES:
        lines.append(f"| {lang} | {measured['fertility'][lang]:.6f} | {measured['unknown_rate'][lang]:.6f} |")
    base = sum(measured["penalised"][lang] for lang in SCORED_LANGUAGES) / 4
    lines += [
        "", f"- Base score: {base:.6f}",
        f"- Guardrail penalty: {measured['guardrail_penalty']:.6f}",
        f"- Reconstruction penalty: {measured['reconstruction_penalty']:.6f}",
        f"- **Full score: {measured['score']:.6f}**", "- Strict reconstruction: 100%",
        f"- Tokenizer SHA-256: `{expected_sha256}`",
        f"- Official checker: `{OFFICIAL_COMMIT}`", "", "## Reproduce", "",
        "Run notebook.ipynb end-to-end. It follows the official starter flow and",
        "trains the selected recipe once, from scratch on train only, then checks",
        "the reloaded file on validation and exports measured results.",
        "The original optimization notebook is preserved at commit 17d34954a86e9baabb46658478ac7a0160e3a04d.",
        "BPE merge ties may vary across retraining runs; always evaluate the generated artifact.", "",
    ]
    (output_dir / "README.md").write_text("\n".join(lines), encoding="utf-8")
    return output_dir


# Save a provenance report outside the team directory.
provenance = {
    "recipe": RECIPE, "boost": BOOST, "min_frequency": MIN_FREQUENCY,
    "dataset": {"id": HF_DATASET, "revision": HF_REVISION,
                "train_rows": len(train), "validation_rows": len(validation),
                "train_fingerprint": train.attrs["dataset_fingerprint"],
                "validation_fingerprint": validation.attrs["dataset_fingerprint"]},
    "training_seconds": training_seconds, "tokenizers_version": tokenizers.__version__,
    "checker_commit": OFFICIAL_COMMIT, "checker_sha256": OFFICIAL_UTILS_SHA256,
    "tokenizer_sha256": measured_sha256, "strict_lossy_rows": 0, "official": report,
}
(RUN_DIR / "validation_report.json").write_text(
    json.dumps(provenance, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
export_dir = export_submission(candidate_path, report, measured_sha256,
                               validation.text.tolist(), RUN_DIR / "export" / TEAM_SLUG)
print("Verified export:", export_dir)
print("Review the result; copy these three files plus this notebook into your team folder.")
try:
    from google.colab import files
except ImportError:
    print("Local run: retrieve the files from the export directory.")
else:
    for filename in ("tokenizer.json", "metadata.yml", "README.md"):
        files.download(str(export_dir / filename))

## Final Checklist

Before opening the PR, check:

- [ ] Changes affect only `submissions/maick-dane-nkou/`.
- [ ] That directory contains only `tokenizer.json`, `metadata.yml`,
      `notebook.ipynb`, and optional `README.md` — no caches, archives or symlinks.
- [ ] The tokenizer loads with `tokenizers==0.22.1`, is at most 20 MiB,
      and has at most 10,000 entries including added tokens.
- [ ] The full score includes both penalties; all six languages were checked.
- [ ] The submitted tokenizer and the reported hash/score describe the same file.
- [ ] The notebook documents training on official **train only** and is valid v4 JSON.
- [ ] The PR targets `main` of the **official AIMS repository**, not your fork.

**About GitHub Actions:** the inspected `Validate submission` workflow checks
the changed team directory, the serialized tokenizer and the evaluator tests.
It does **not** execute this notebook or require its headings to match the
starter. A PR touching `submissions/**` is a trigger even when the source branch
is not named `submission`; the fork's push trigger is restricted to `submission`.
First-time contributors may need an organizer to approve workflow execution.
Notebook formatting cannot guarantee runner availability, permissions or test success.

Do not modify `.github/workflows/`, `starter/`, evaluation code or the leaderboard
in a competition-entry PR. Nightly scoring is a separate organizer workflow;
local validation is not an official hidden-test leaderboard result.

# References

- [Official starter notebook](https://github.com/aims-ai-research-foundations/airf-multilingual-tokenizer-challenge/blob/main/starter/starter.ipynb)
- [Competition rules and submission instructions](https://github.com/aims-ai-research-foundations/airf-multilingual-tokenizer-challenge/blob/main/CONTRIBUTING.md)
- [Public dataset](https://huggingface.co/datasets/Similoluwa/african-multilingual-tokenizer-challenge), revision `v1.0.0`
- [Pinned official checker](https://github.com/aims-ai-research-foundations/airf-multilingual-tokenizer-challenge/blob/75578f2400c39b1f8e31ce7e7104b37fbc470d11/starter/utils.py)
- [Original optimization notebook and artifact](https://github.com/maick-code/airf-multilingual-tokenizer-challenge/tree/17d34954a86e9baabb46658478ac7a0160e3a04d/submissions/maick-dane-nkou)
- [AI Research Foundations learning path](https://www.skills.google/paths/3135)